# P4c — Clinical BERT Models (M3 and M4)

## 1. Introduction and Objective

In Notebook 04b, both TF-IDF models achieved modest improvements over the regex baseline, with the best validation F1 score reaching 0.6960. These results suggest that simple lexical patterns already capture a substantial portion of the predictive information in ED chief complaints. I now examine whether pretrained contextual representations can provide additional predictive value.

Inspired by the application of contextual language models to ED chief complaint classification by Chang et al. (2020), I use Bio_ClinicalBERT (Alsentzer et al., 2019) to develop two classifiers. M3 uses frozen BERT embeddings as input to an L2-regularized logistic regression classifier, while M4 fine-tunes the pretrained BERT model directly on the LBP classification task.

By using the same pretrained checkpoint and experimental partitions, I examine how frozen feature extraction and task-specific fine-tuning perform relative to each other and to the lexical models developed in the previous notebooks.

## 2. Development Dataset Preparation

I use the same modeling dataset and fixed patient-grouped partitions as in the previous notebooks to ensure a consistent comparison across models.

Both M3 and M4 use the cleaned chief complaint text as input and the ICD-derived binary LBP label as the prediction target. The training set is used for model development, and the validation set is used for model selection and classification threshold selection. The test set remains held out for the final comparative evaluation.

In [1]:
from pathlib import Path
import pandas as pd

# Set the data directory for the current environment
DATA_DIR = Path("../results/03_experimental_setup")

# Load the modeling dataset and fixed partition assignments
df = pd.read_csv(
    DATA_DIR / "modeling_dataset.csv",
    keep_default_na=False
)

splits = pd.read_csv(DATA_DIR / "dataset_splits.csv")

# Retain development partitions only
dev_splits = splits[splits["split"].isin(["train", "validation"])]

dev_df = df.merge(
    dev_splits,
    on="stay_id",
    how="inner",
    validate="one_to_one"
)

train_df = dev_df[dev_df["split"] == "train"].copy().reset_index(drop=True)
val_df = dev_df[dev_df["split"] == "validation"].copy().reset_index(drop=True)

# Prepare model inputs and reference labels
X_train = train_df["chiefcomplaint_clean"]
y_train = train_df["lbp_label"]

X_val = val_df["chiefcomplaint_clean"]
y_val = val_df["lbp_label"]

# Check input integrity
assert len(dev_df) == len(dev_splits)
assert dev_df["chiefcomplaint_clean"].notna().all()
assert dev_df["chiefcomplaint_clean"].str.strip().ne("").all()
assert dev_df["lbp_label"].isin([0, 1]).all()

# Summarize development partitions
summary = dev_df.groupby("split").agg(
    encounters=("stay_id", "size"),
    patients=("subject_id", "nunique"),
    positives=("lbp_label", "sum")
)

summary["negatives"] = summary["encounters"] - summary["positives"]
summary["prevalence (%)"] = (
    summary["positives"] / summary["encounters"] * 100
).round(2)

display(summary)

,encounters,patients,positives,negatives,prevalence (%)
split,,,,,
train,295877,143119,11429,284448,3.86
validation,42268,20483,1632,40636,3.86


## 3. Clinical BERT Configuration and Pilot

I use the pretrained Bio_ClinicalBERT checkpoint and its original tokenizer for both M3 and M4. Given the short length of ED chief complaints, I set the maximum sequence length to 64 tokens and use dynamic padding during batch processing to avoid unnecessary computation.

To support execution across different computing environments, I automatically select CUDA, MPS, or CPU based on device availability. Before processing the full dataset, I run a small pilot to verify tokenization, model output dimensions, and computational feasibility.

In [2]:
import random
import numpy as np
import torch
import transformers
from time import perf_counter
from transformers import AutoTokenizer, AutoModel

# Set random seeds
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Select the available computing device
if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.cuda.manual_seed_all(SEED)
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

# Enable mixed precision on CUDA
use_fp16 = device.type == "cuda"

print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"Device: {device}")
print(f"FP16 mixed precision: {use_fp16}")

if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.15.0
Transformers version: 5.24.1
Device: mps
FP16 mixed precision: False


In [3]:
%pip install ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [4]:
# Shared model configuration
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
MAX_LENGTH = 64
BATCH_SIZE = 32

# Load the pretrained tokenizer and encoder
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME).to(device)

# Freeze the encoder for M3
encoder.requires_grad_(False)
encoder.eval()

# Summarize model configuration
print(f"Checkpoint: {MODEL_NAME}")
print(f"Vocabulary size: {len(tokenizer):,}")
print(f"Hidden size: {encoder.config.hidden_size}")
print(f"Maximum sequence length: {MAX_LENGTH}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Total parameters: {sum(p.numel() for p in encoder.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in encoder.parameters() if p.requires_grad):,}")

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

W1127 00:15:36.667000 30025 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  436MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

model.safetensors: downloading bytes:           |  0.00B            

Checkpoint: emilyalsentzer/Bio_ClinicalBERT
Vocabulary size: 28,996
Hidden size: 768
Maximum sequence length: 64
Batch size: 32
Total parameters: 108,310,272
Trainable parameters: 0


In [5]:
from contextlib import nullcontext

# Synthetic examples for the initial sanity check
sample_texts = [
    "back pain",
    "lower back pain",
    "LBP",
    "abdominal pain",
    "severe back pain after a fall"
]

# Tokenize with dynamic padding
inputs = tokenizer(
    sample_texts,
    padding=True,
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors="pt"
)

inputs = {k: v.to(device) for k, v in inputs.items()}

# Run a forward pass through the frozen encoder
with torch.inference_mode():
    with torch.autocast("cuda", dtype=torch.float16) if use_fp16 else nullcontext():
        outputs = encoder(**inputs)

    cls_embeddings = outputs.last_hidden_state[:, 0, :]

# Check output dimensions
print(f"Tokenized input shape: {inputs['input_ids'].shape}")
print(f"Hidden state shape: {outputs.last_hidden_state.shape}")
print(f"CLS embedding shape: {cls_embeddings.shape}")

assert cls_embeddings.shape == (len(sample_texts), encoder.config.hidden_size)
assert torch.isfinite(cls_embeddings).all().item()

print("Forward pass sanity check: PASSED")

: 

In [ ]:
# Select a small sample from the training set
pilot_texts = X_train.sample(
    n=1024,
    random_state=SEED
).tolist()

def synchronize_device():
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        torch.mps.synchronize()

# Measure embedding extraction throughput
synchronize_device()
start = perf_counter()

with torch.inference_mode():
    for i in range(0, len(pilot_texts), BATCH_SIZE):
        batch = pilot_texts[i:i + BATCH_SIZE]

        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.autocast("cuda", dtype=torch.float16) if use_fp16 else nullcontext():
            outputs = encoder(**inputs)

        embeddings = outputs.last_hidden_state[:, 0, :].float().cpu()

synchronize_device()
elapsed = perf_counter() - start

throughput = len(pilot_texts) / elapsed
estimated_minutes = (len(X_train) + len(X_val)) / throughput / 60

print(f"Pilot samples: {len(pilot_texts):,}")
print(f"Elapsed time: {elapsed:.2f} seconds")
print(f"Throughput: {throughput:.1f} encounters/second")
print(f"Estimated full embedding extraction: {estimated_minutes:.1f} minutes")